# 오늘춥나?? — 2단계: 안양대 날씨 예측 모델 학습

`train_data.csv`로 **"만안(377) 날씨 → 안양대 날씨"** 관계를 배우고, 모델 세 개를 비교해요.

- **문제(입력):** 만안 기온·습도·풍속 + 월 + 시간대
- **정답:** 안양대 500m 격자 기온·습도·풍속
- **학습:** 2023~2024년 / **시험:** 2025년 (모델이 한 번도 못 본 해로 채점)
- **비교 모델:** ① 평균 편차(기준선) ② 선형 회귀 ③ 랜덤포레스트
- **최종 채점:** 예측한 세 값으로 기상청 체감온도를 계산해서, 실제 체감온도와 몇 도 차이 나는지

**순서:** ① 불러오기 → ② 나누기 → ③ 체감온도 공식 → ④ 모델 3개 학습 → ⑤ 채점 → ⑥ 랜덤포레스트가 본 중요도 → ⑦ 저장 → ⑧ 써보기

## ① 드라이브에서 불러오기

In [ ]:
import numpy as np
import pandas as pd
import joblib
from google.colab import drive
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

drive.mount("/content/drive")
BASE = "/content/drive/MyDrive/onulchupna"
data = pd.read_csv(f"{BASE}/train_data.csv", parse_dates=["tm"])
print(len(data), "줄,", data.tm.min(), "~", data.tm.max())
data.head()

## ② 학습용 / 시험용 나누기

**연도로** 나눠요. 줄을 무작위로 섞으면 1시는 학습, 2시는 시험에 들어가서 답을 미리 본 것처럼 점수가 부풀려져요.

시간대와 월은 순환해요(23시 다음이 0시, 12월 다음이 1월). 그래서 sin·cos로 바꿔서 23시와 0시가 가깝다는 걸 모델이 알게 해요.

In [ ]:
TEST_YEAR = 2025

def add_features(df):
    df = df.copy()
    df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
    df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)
    df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12)
    df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12)
    return df

data = add_features(data)
FEATURES = ["aws_ta", "aws_hm", "aws_ws", "hour_sin", "hour_cos", "month_sin", "month_cos"]
TARGETS = ["uni_ta", "uni_hm", "uni_ws"]

train = data[data.tm.dt.year < TEST_YEAR].reset_index(drop=True)
test = data[data.tm.dt.year == TEST_YEAR].reset_index(drop=True)
print(f"학습 {len(train)}줄 ({train.tm.min():%Y-%m} ~ {train.tm.max():%Y-%m})")
print(f"시험 {len(test)}줄 ({test.tm.min():%Y-%m} ~ {test.tm.max():%Y-%m})")

## ③ 기상청 체감온도 공식

- 겨울(10~4월): 기온 10°C 이하이고 풍속 1.3 m/s 이상일 때만 바람 공식, 아니면 기온 그대로
- 여름(5~9월): 습도 공식 (습구온도는 Stull 추정식)

In [ ]:
def kma_winter(ta, ws):
    v = np.asarray(ws) * 3.6
    wc = 13.12 + 0.6215 * ta - 11.37 * v**0.16 + 0.3965 * v**0.16 * ta
    return np.where((np.asarray(ta) <= 10) & (np.asarray(ws) >= 1.3), wc, ta)

def kma_summer(ta, rh):
    tw = (ta * np.arctan(0.151977 * np.sqrt(rh + 8.313659)) + np.arctan(ta + rh)
          - np.arctan(rh - 1.67633) + 0.00391838 * rh**1.5 * np.arctan(0.023101 * rh) - 4.686035)
    return -0.2442 + 0.55399 * tw + 0.45535 * ta - 0.0022 * tw**2 + 0.00278 * tw * ta + 3.0

def kma_feels(month, ta, rh, ws):
    month, ta, rh, ws = (np.asarray(x, dtype=float) for x in (month, ta, rh, ws))
    return np.where((month >= 5) & (month <= 9), kma_summer(ta, rh), kma_winter(ta, ws))

def tidy(pred):
    """예측값을 말이 되는 범위로: 습도 0~100%, 풍속 0 이상"""
    pred = np.array(pred, dtype=float)
    pred[:, 1] = pred[:, 1].clip(0, 100)
    pred[:, 2] = pred[:, 2].clip(0, None)
    return pred

## ④ 모델 세 개 학습

모델은 안양대 값을 통째로 맞히지 않고, **만안과의 차이(보정량)**를 배워요. 예측할 때는 `만안 값 + 보정량`으로 계산해요.
- 처음부터 맞히게 하면 습도처럼 숫자 폭이 큰 값에 모델이 끌려가서 기온 예측이 망가지기 쉬워요.
- 차이를 배우면 처음 보는 더위·추위가 와도 만안 값을 기준으로 따라가요.
- 기온·습도·풍속은 **각각 따로** 모델을 만들어요.

1. **평균 편차 (기준선):** 월 × 시간대마다 '안양대 − 만안' 평균 차이를 더해요.
2. **선형 회귀:** 입력과 보정량을 직선 관계로 배워요.
3. **랜덤포레스트:** 결정트리 300그루의 예측을 평균내요.

In [ ]:
BASES = ["aws_ta", "aws_hm", "aws_ws"]   # 각 정답의 기준이 되는 만안 값

class MeanBias:
    """월 × 시간대별 평균 차이"""
    def fit(self, df):
        d = pd.DataFrame({t: df[t] - df[b] for t, b in zip(TARGETS, BASES)})
        d["month"], d["hour"] = df["month"], df["hour"]
        self.table = d.groupby(["month", "hour"])[TARGETS].mean()
        self.overall = self.table.mean()
        return self
    def predict(self, df):
        t = self.table.reindex(pd.MultiIndex.from_arrays([df.month, df.hour])).fillna(self.overall).to_numpy()
        return tidy(df[BASES].to_numpy(dtype=float) + t)

class DeltaModel:
    """정답마다 따로: (안양대 − 만안) 보정량을 학습, 예측은 만안 + 보정량"""
    def __init__(self, make):
        self.make = make
    def fit(self, df):
        self.models = {t: self.make().fit(df[FEATURES], df[t] - df[b]) for t, b in zip(TARGETS, BASES)}
        return self
    def predict(self, df):
        delta = np.column_stack([self.models[t].predict(df[FEATURES]) for t in TARGETS])
        return tidy(df[BASES].to_numpy(dtype=float) + delta)

make_rf = lambda: RandomForestRegressor(n_estimators=300, min_samples_leaf=5, n_jobs=-1, random_state=0)
models = {
    "평균 편차": MeanBias().fit(train),
    "선형 회귀": DeltaModel(LinearRegression).fit(train),
    "랜덤포레스트": DeltaModel(make_rf).fit(train),
}

def predict(name, df):
    return models[name].predict(df)

print("학습 완료:", list(models))

## ⑤ 2025년으로 채점

숫자는 **평균 절대 오차(MAE)**예요. 예측이 실제와 평균 몇 도(%, m/s) 차이 나는지를 뜻하고, **작을수록 좋아요.**

- `보정 안 함`: 만안 값을 그대로 안양대 값이라고 했을 때. 날씨앱만 믿는 것과 같아요.
- 맨 오른쪽 `체감온도`가 최종 목표 점수예요.

In [ ]:
real_feels = kma_feels(test.month, test.uni_ta, test.uni_hm, test.uni_ws)

def score(pred):
    p_feels = kma_feels(test.month, pred[:, 0], pred[:, 1], pred[:, 2])
    return {
        "기온 오차(°C)": np.abs(pred[:, 0] - test.uni_ta).mean(),
        "습도 오차(%)": np.abs(pred[:, 1] - test.uni_hm).mean(),
        "풍속 오차(m/s)": np.abs(pred[:, 2] - test.uni_ws).mean(),
        "체감온도 오차(°C)": np.abs(p_feels - real_feels).mean(),
    }

rows = {"보정 안 함": score(test[["aws_ta", "aws_hm", "aws_ws"]].to_numpy(dtype=float))}
for name in models:
    rows[name] = score(predict(name, test))
result = pd.DataFrame(rows).T.round(2)
best = result["체감온도 오차(°C)"].idxmin()
print("체감온도를 가장 잘 맞힌 모델:", best)
result

## ⑤-2 적중률(%)로 보기

회귀 모델은 정답이 숫자라서, **"예측 체감온도가 실제와 ±N°C 안에 들어오면 적중"**으로 정해 비율을 봐요.
- ±0.5°C: 아주 정확 / ±1°C: 사람이 거의 구분 못 하는 차이 / ±2°C: 옷차림이 달라지기 전 수준
- 계절별로도 나눠서, 겨울·여름 중 어디서 더 잘 맞히는지 봐요.

In [ ]:
TOLERANCES = [0.5, 1.0, 2.0]   # 허용 오차(°C)
season = np.where(test.month.isin([12, 1, 2]), "겨울",
         np.where(test.month.isin([3, 4, 5]), "봄",
         np.where(test.month.isin([6, 7, 8]), "여름", "가을")))

preds = {"보정 안 함": test[["aws_ta", "aws_hm", "aws_ws"]].to_numpy(dtype=float)}
preds.update({name: predict(name, test) for name in models})

acc_rows, season_rows = {}, {}
for name, pred in preds.items():
    err = np.abs(kma_feels(test.month, pred[:, 0], pred[:, 1], pred[:, 2]) - real_feels)
    acc_rows[name] = {f"±{t}°C 적중률(%)": (err <= t).mean() * 100 for t in TOLERANCES}
    season_rows[name] = pd.Series(err <= 1.0).groupby(season).mean().mul(100)

accuracy = pd.DataFrame(acc_rows).T.round(1)
print("체감온도 적중률 (2025년 시험, 높을수록 좋음)")
display(accuracy)

print("계절별 ±1°C 적중률(%)")
display(pd.DataFrame(season_rows).T[["봄", "여름", "가을", "겨울"]].round(1))

best = accuracy["±1.0°C 적중률(%)"].idxmax()
base = accuracy.loc["보정 안 함", "±1.0°C 적중률(%)"]
print(f"→ {best}: 체감온도를 ±1°C 안으로 맞힌 비율 {accuracy.loc[best, '±1.0°C 적중률(%)']:.1f}%"
      f" (보정 안 했을 때 {base:.1f}%)")

## ⑥ 랜덤포레스트는 무엇을 중요하게 봤을까

정답(기온·습도·풍속 보정량)마다 어떤 입력을 많이 썼는지 보여줘요. 값이 클수록 중요해요.

In [ ]:
rf = models["랜덤포레스트"]
pd.DataFrame({t: rf.models[t].feature_importances_ for t in TARGETS}, index=FEATURES).round(3)

## ⑦ 최종 모델 저장

시험이 끝났으니, **2025년까지 전체 데이터로 다시 학습**한 랜덤포레스트를 드라이브에 저장해요. 사이트에 붙일 때 이 파일을 써요.

(⑤에서 다른 모델이 더 좋았다면 결과를 알려주세요. 그 모델로 저장하도록 바꿔드릴게요.)

In [ ]:
final = DeltaModel(make_rf).fit(data)
joblib.dump({"models": final.models, "features": FEATURES, "targets": TARGETS, "bases": BASES},
            f"{BASE}/campus_model.joblib", compress=3)
print("저장:", f"{BASE}/campus_model.joblib")

## ⑧ 써보기: 예보값을 넣어 안양대 체감온도 예측

실제로는 여기에 **만안구 단기예보**를 넣어요. 아래 숫자를 바꿔보세요.

In [ ]:
def campus_forecast(when, ta, hm, ws):
    when = pd.Timestamp(when)
    x = add_features(pd.DataFrame({"aws_ta": [ta], "aws_hm": [hm], "aws_ws": [ws],
                                   "month": [when.month], "hour": [when.hour]}))
    p = final.predict(x)[0]
    feels = float(kma_feels(when.month, p[0], p[1], p[2]))
    app = float(kma_feels(when.month, ta, hm, ws))
    print(f"{when:%m월 %d일 %H시} 예보: 기온 {ta}°C, 습도 {hm}%, 풍속 {ws} m/s → 날씨앱 체감 {app:.1f}°C")
    print(f"  안양대 예측: 기온 {p[0]:.1f}°C, 습도 {p[1]:.0f}%, 풍속 {p[2]:.1f} m/s → 캠퍼스 체감 {feels:.1f}°C")

campus_forecast("2026-12-01 08:00", ta=-3.0, hm=60, ws=2.5)
campus_forecast("2026-07-15 14:00", ta=31.0, hm=75, ws=1.0)